# Day 4 & 5: Silver Layer Transformation
Requirement: Transform Bronze -> Silver: clean nulls, dedupe, standardize column names.
Manage malformed records -> quarantine.
Develop User-Defined Functions (UDFs) using Pandas/Python UDF to standardize DataFrame/Table headers
Apply business rules (currency normalization, date standardization).
Demonstrate Delta Lake ACID, time travel.

In [0]:
%run "/Workspace/Users/sanskruti.r.sampate@v4c.ai/vstone/src/notebooks/00_configs"

In [0]:
dbutils.widgets.dropdown(
    "bronze_table",
    "flight_bronze",
    [
        "flight_bronze",
        "flight_json_bronze",
        "flight_xml_bronze",
        "flight_csv_incremental_bronze"
    ]
)

bronze_table = dbutils.widgets.get("bronze_table")

In [0]:
silver_table = TABLE_MAPPING[bronze_table]["silver"]
quarantine_table = TABLE_MAPPING[bronze_table]["quarantine"]

In [0]:
bronze_table_full_name = f"{catalog}.{bronze_schema}.{bronze_table}"
silver_table_full_name = f"{catalog}.{silver_schema}.{silver_table}"
quarantine_table_full_name = f"{catalog}.{quarantine_schema}.{quarantine_table}"

In [0]:
print(bronze_table_full_name)
print(silver_table_full_name)
print(quarantine_table_full_name)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    trim,
    upper,
    current_timestamp,
    lit
)

from pyspark.sql.types import StringType

from pyspark.sql.functions import pandas_udf

### 1. Read from Bronze

In [0]:
bronze_df = spark.read.table(bronze_table_full_name)
print (f"records : {bronze_df.count()}")
print (f"Columns : {len(bronze_df.columns)}")
bronze_df.printSchema()
display(bronze_df.limit(5))


### 2. Pandas UDF to Standardize Column Names
Requirement: Develop User-Defined Functions (UDFs) using Pandas/Python UDF to standardize DataFrame/Table headers.
Since column names are metadata, we usually rename them via PySpark `withColumnRenamed` or `toDF`. However, to fulfill the requirement of using a Pandas UDF, we will demonstrate applying a Pandas UDF on string data (e.g., standardizing city names) and standardizing headers using standard PySpark list comprehension.

# User Defined Function (UDF)

## Purpose

This Pandas UDF is used to standardize textual columns before loading data into the Silver Layer.

## Operations Performed

- Removes leading and trailing spaces
- Converts values to uppercase
- Ensures consistent formatting across datasets

## Applied On

- Airline
- Origin Airport
- Destination Airport
- State Names

## Benefits

- Improves data quality
- Enables accurate joins
- Prevents duplicate values caused by inconsistent casing


In [0]:
# Standardize headers
standardized_columns = [
    col.lower().replace(" ", "_")
    for col in bronze_df.columns
]

df_standard_headers = bronze_df.toDF(*standardized_columns)

print("Column names standardized.")

In [0]:
@pandas_udf("string")
def standardize_string_udf(s):
    return s.str.upper().str.strip()

In [0]:
from pyspark.sql import functions as F

string_columns = [
    "airline",
    "origin",
    "dest",
    "origincityname",
    "destcityname",
    "originstate",
    "deststate"
]

for c in string_columns:
    if c in df_standard_headers.columns:
        df_standard_headers = (
            df_standard_headers
            .withColumn(c, standardize_string_udf(F.col(c)))
        )

display(df_standard_headers.limit(5))

%md
### 3. Apply Business Rules and Clean Data
- Clean nulls (drop or impute)
- Dedupe
- Date standardization (e.g., YYYY-MM-DD)
- Currency normalization

In [0]:
business_keys = [
    "flightdate",
    "airline",
    "origin",
    "dest",
    "crsdeptime"
]

existing_keys = [
    c for c in business_keys
    if c in df_standard_headers.columns
]

df_deduped = df_standard_headers.dropDuplicates(existing_keys)

print("Before:", df_standard_headers.count())
print("After :", df_deduped.count())

In [0]:
df_cleaned = df_deduped

# Standardize Flight Date
if "flightdate" in df_cleaned.columns:
    df_cleaned = df_cleaned.withColumn(
        "flightdate",
        F.to_date("flightdate")
    )

# Round numeric columns
numeric_columns = [
    "distance",
    "airtime",
    "depdelayminutes",
    "arrdelayminutes"
]

for c in numeric_columns:
    if c in df_cleaned.columns:
        df_cleaned = df_cleaned.withColumn(
            c,
            F.round(F.col(c), 2)
        )

# Audit Columns
df_cleaned = (
    df_cleaned
    .withColumn("processed_timestamp", F.current_timestamp())
    .withColumn("pipeline_layer", F.lit("Silver"))
)

display(df_cleaned.limit(5))

In [0]:
display(
    df_cleaned.select(
        F.count(F.when(F.col("distance") < 0, 1)).alias("negative_distance"),
        F.count(F.when(F.col("airtime") < 0, 1)).alias("negative_airtime")
    )
)

### 4. Quarantine Malformed Records
Records with critical nulls (e.g., null origin city) go to Quarantine.

In [0]:
from pyspark.sql import functions as F

quarantine_condition = F.lit(False)

# Critical columns
critical_columns = [
    "flightdate",
    "airline",
    "origin",
    "dest"
]

for c in critical_columns:
    if c in df_cleaned.columns:
        quarantine_condition = (
            quarantine_condition |
            F.col(c).isNull()
        )

# Invalid numeric values
if "distance" in df_cleaned.columns:
    quarantine_condition = (
        quarantine_condition |
        (
            F.col("distance").isNotNull() &
            (F.col("distance") < 0)
        )
    )

if "airtime" in df_cleaned.columns:
    quarantine_condition = (
        quarantine_condition |
        (
            F.col("airtime").isNotNull() &
            (F.col("airtime") < 0)
        )
    )

good_records_df = df_cleaned.filter(~quarantine_condition)
quarantined_records_df = df_cleaned.filter(quarantine_condition)

print("Good :", good_records_df.count())
print("Quarantine :", quarantined_records_df.count())
print("Total :", good_records_df.count() + quarantined_records_df.count())
print("Original :", df_cleaned.count())

### 5. Write to Silver and Quarantine Tables

In [0]:
good_records_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(silver_table_full_name)

In [0]:
quarantined_records_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(quarantine_table_full_name)

In [0]:
silver_df = spark.read.table(silver_table_full_name)

print("=" * 50)
print("Bronze Records :", bronze_df.count())
print("Silver Records :", silver_df.count())
print("Quarantine Records :", quarantined_records_df.count())
print("Good Records :", good_records_df.count())

assert silver_df.count() == good_records_df.count()

print("✅ Validation Passed")

In [0]:
print("Bronze Columns :", len(bronze_df.columns))
print("Silver Columns :", len(silver_df.columns))

silver_df.printSchema()

In [0]:
spark.sql(f"OPTIMIZE {silver_table_full_name}")

In [0]:
spark.sql(f"""
COMMENT ON TABLE {silver_table_full_name}
IS 'Silver layer containing cleaned, standardized and validated flight delay data.'
""")

In [0]:
%sql
select * from vstone.gold.airport_delay_summary